In [1]:
pip show tensorflow

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install tensorflow

  Using cached tensorflow-2.17.0-cp311-cp311-win_amd64.whl.metadata (3.2 kB)
  Using cached tensorflow_intel-2.17.0-cp311-cp311-win_amd64.whl.metadata (5.0 kB)
Using cached tensorflow-2.17.0-cp311-cp311-win_amd64.whl (2.0 kB)
Using cached tensorflow_intel-2.17.0-cp311-cp311-win_amd64.whl (385.0 MB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\Admin\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python311\\site-packages\\tensorflow\\include\\external\\com_github_grpc_grpc\\src\\core\\ext\\filters\\client_channel\\lb_policy\\grpclb\\client_load_reporting_filter.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: C:\Users\Admin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import sys
print(sys.executable)

C:\Users\Admin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe


In [4]:
import tensorflow as tf

ModuleNotFoundError: No module named 'tensorflow.python'

In [1]:
import cv2
import numpy as np
import time
import threading

# Load the known face image
known_face_image = cv2.imread("1234.jpg", cv2.IMREAD_GRAYSCALE)

# Extract the contours of the known face image
contours_known, _ = cv2.findContours(known_face_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Initialize the camera
cap = cv2.VideoCapture(0)

# Set the exam duration
exam_duration = 2 * 60 * 60  # 2 hours

# Set the threshold matching
threshold = 0.8

# Set the time interval for authorization
time_interval = 15 * 60  # 15 minutes

start_time = time.time()

# Create two separate windows
cv2.namedWindow("Candidate")
cv2.namedWindow("Authorization")

# Create a lock for synchronization
camera_lock = threading.Lock()

# Create a queue for frames
frame_queue = []

# Create a condition variable to signal when a frame is available
frame_available = threading.Condition()

def capture_and_display_frames():
    while True:
        try:
            with camera_lock:
                ret, frame = cap.read()
            if not ret:
                break
            cv2.imshow("Candidate", frame)
            with frame_available:
                frame_queue.append(frame)
                frame_available.notify_all()
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        except Exception as e:
            print("Error capturing frame:", e)

def authorization_logic():
    while True:
        try:
            with frame_available:
                while len(frame_queue) == 0:
                    frame_available.wait()
                frame = frame_queue.pop(0)
            
            # Convert the frame to grayscale
            gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            # Extract the contours of the frame
            contours_frame, _ = cv2.findContours(gray_frame, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            # Check if contours were found in both the known face and the frame
            if len(contours_known) == 0 or len(contours_frame) == 0:
                print("No contours found")
                cv2.imshow("Candidate", cv2.putText(frame, "No contours found", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2))
                continue

            # Calculate the similarity between contours
            similarity = 0
            for contour_known in contours_known:
                for contour_frame in contours_frame:
                    similarity += cv2.matchShapes(contour_known, contour_frame, cv2.CONTOURS_MATCH_I1, 0)

            # Calculate the average similarity
            avg_similarity = similarity / (len(contours_known) * len(contours_frame))

            # Check if the average similarity is above the threshold
            if avg_similarity > threshold:
                print("Authorized")

                # Display the authorization camera frame for 15 minutes
                start_auth_time = time.time()
                while time.time() - start_auth_time < time_interval:
                    if len(frame_queue) > 0:
                        auth_frame = frame_queue.pop(0)
                        cv2.imshow("Authorization", auth_frame)
                        if cv2.waitKey(1) & 0xFF == ord('q'):
                            break
                    else:
                        time.sleep(0.1)

            else:
                # Display a message indicating that the candidate is not authorized
                cv2.imshow("Candidate", cv2.putText(frame, "Not Authorized", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2))

            # Check if the exam duration has been reached
            if time.time() - start_time > exam_duration:
                break
        except Exception as e:
            print("Error processing frame:", e)

# Start the threads
frame_thread = threading.Thread(target=capture_and_display_frames)
frame_thread.daemon = True
frame_thread.start()

auth_thread = threading.Thread(target=authorization_logic)
auth_thread.daemon = True
auth_thread.start()

# Wait for the threads to finish
frame_thread.join()
auth_thread.join()

# Release resources
cap.release()
cv2.destroyAllWindows()